# 09n FastReID SBS R50-IBN Fine-Tune on CityFlowV2

Fine-tune the FastReID SBS ResNet50-IBN-a checkpoint on CityFlowV2 by downloading the raw benchmark from Google Drive inside Kaggle, preparing ReID crops on the fly, then exporting the best checkpoint for downstream MTMC ensemble experiments.

## Recipe
1. Clone the project from `feature/pretrained-ensemble`
2. Install notebook dependencies, including `gdown` for CityFlowV2 download
3. Download CityFlowV2 from Google Drive and flatten the train/validation camera directories
4. Extract crops from video frames using `gt.txt` and build `train/query/gallery` splits under `/tmp/cityflowv2_crops`
5. Download the VeRi SBS R50-IBN weights from the FastReID GitHub release
6. Remap FastReID checkpoint keys into `ReIDModelResNet50IBN` from `src/training/model.py`
7. Train with GeM + BNNeck, PK sampling (`16 IDs x 4 instances`), AdamW, cosine annealing, and mixed precision
8. Evaluate every 10 epochs with cosine-distance mAP / Rank-1 on the prepared CityFlowV2 query/gallery split and save the best `state_dict`

## Outputs
- `/kaggle/working/fastreid_r50_ibn_cityflowv2_best.pth`
- `/kaggle/working/09n_fastreid_r50_finetune/training_history.json`
- `/kaggle/working/09n_fastreid_r50_finetune/final_metrics.json`

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

!pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124 -q
pip_install("loguru", "omegaconf", "scikit-learn", "matplotlib", "gdown")

import torch

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
runtime_summary = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "gpus": gpu_names,
}
print(json.dumps(runtime_summary, indent=2))

In [ ]:
import os
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
PROJECT = WORK_DIR / 'gp'
REPO_URL = 'https://github.com/MRKDaGods/gp.git'
BRANCH = 'feature/pretrained-ensemble'

if not PROJECT.exists():
    print(f'Cloning {REPO_URL} [{BRANCH}] -> {PROJECT}')
    subprocess.check_call([
        'git',
        'clone',
        '--depth',
        '1',
        '--branch',
        BRANCH,
        REPO_URL,
        str(PROJECT),
    ])
else:
    print(f'Project already present at {PROJECT}')

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

requirements_path = PROJECT / 'requirements.txt'
if requirements_path.exists():
    pip_install('-r', str(requirements_path))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps', '-q'])

try:
    __import__('torchreid')
except ImportError:
    pip_install('git+https://github.com/KaiyangZhou/deep-person-reid.git')

print(json.dumps({
    'cwd': os.getcwd(),
    'project': str(PROJECT),
    'branch': BRANCH,
    'requirements_found': requirements_path.exists(),
}, indent=2))

## 2. CityFlowV2 Download + ReID Split Prep

This notebook prepares CityFlowV2 inside Kaggle by downloading the archive from Google Drive, extracting crops from `gt.txt`, and building nested `train/query/gallery` splits under `/tmp/cityflowv2_crops` before any model setup runs.

In [ ]:
import random
import urllib.request
import zipfile
from collections import defaultdict

import cv2
import numpy as np

OUTPUT_DIR = Path("/kaggle/working/09n_fastreid_r50_finetune")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
BEST_MODEL_PATH = Path("/kaggle/working/fastreid_r50_ibn_cityflowv2_best.pth")
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_checkpoint.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
METRICS_PATH = OUTPUT_DIR / "final_metrics.json"
for directory in [OUTPUT_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CITYFLOW_DIR = Path("/tmp/cityflowv2")
RAW_CROP_DIR = Path("/tmp/cityflowv2_raw_crops")
REID_ROOT = Path("/tmp/cityflowv2_crops")
GDRIVE_ID = "13wNJpS_Oaoe-7y5Dzexg_Ol7bKu1OWuC"
ARCHIVE_NAME = "AIC22_Track1_MTMC_Tracking.zip"
ALLOWED_SPLITS = {"train", "validation"}
MAX_CROPS_PER_ID_CAM = 20
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
QUERY_CAMS_PER_ID = 2
QUERY_FRAMES_PER_CAM = 10
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMG_SIZE = (256, 256)
BATCH_SIZE = 64
P_IDS = 16
K_INSTANCES = 4
NUM_WORKERS = 4
EPOCHS = 200
WARMUP_EPOCHS = 10
FREEZE_BACKBONE_EPOCHS = 5
EVAL_EVERY = 10

BACKBONE_LR = 3e-5
HEAD_LR = 5e-4
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
TRIPLET_MARGIN = 0.3
TRIPLET_WEIGHT = 0.3
CENTER_WEIGHT = 5e-4
CENTER_START_EPOCH = 15
GEM_P = 3.0
NUM_EXPECTED_CLASSES = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE.startswith("cuda")

SBS_WEIGHTS_URL = "https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth"
SBS_WEIGHTS_PATH = OUTPUT_DIR / "veri_sbs_R50-ibn.pth"


def split_counts(root: Path) -> dict[str, int]:
    counts = {}
    for split in ["train", "query", "gallery"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    return counts


def prepared_reid_exists(root: Path) -> bool:
    counts = split_counts(root)
    return all(counts[split] > 0 for split in ["train", "query", "gallery"])


def find_gt(cam_dir: Path) -> Path | None:
    direct = cam_dir / "gt.txt"
    if direct.exists():
        return direct
    nested = cam_dir / "gt" / "gt.txt"
    if nested.exists():
        return nested
    matches = list(cam_dir.rglob("gt.txt"))
    return matches[0] if matches else None


def find_video(cam_dir: Path) -> Path | None:
    for ext in ["avi", "mp4"]:
        for stem in ["vdo", "video"]:
            candidate = cam_dir / f"{stem}.{ext}"
            if candidate.exists():
                return candidate
    matches = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
    return matches[0] if matches else None


def ensure_cityflow_download() -> Path:
    global CITYFLOW_DIR

    preferred_roots = [
        CITYFLOW_DIR,
        Path("/kaggle/input/cityflowv2"),
        Path("/kaggle/input/aic22-track1-mtmc-tracking"),
    ]
    for check_dir in preferred_roots:
        if check_dir.exists() and next(check_dir.rglob("vdo.avi"), None) is not None:
            print(f"CityFlowV2 already available at {check_dir}")
            CITYFLOW_DIR = check_dir
            return check_dir

    CITYFLOW_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = Path(f"/tmp/{ARCHIVE_NAME}")
    if not archive_path.exists():
        print(f"Downloading CityFlowV2 from Google Drive (id={GDRIVE_ID})...")
        import gdown

        gdown.download(f"https://drive.google.com/uc?id={GDRIVE_ID}", str(archive_path), quiet=False)
    else:
        print(f"Archive already downloaded: {archive_path}")

    staging = Path("/tmp/_aic22_staging")
    staging.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path} to {staging}...")
    with zipfile.ZipFile(str(archive_path), "r") as zf:
        zf.extractall(str(staging))

    if archive_path.exists():
        archive_path.unlink()
        print("Deleted archive to reclaim space.")

    moved = 0
    skipped_splits = set()
    for vdo_path in sorted(staging.rglob("vdo.avi")):
        cam_dir = vdo_path.parent
        scene_dir = cam_dir.parent
        split_dir = scene_dir.parent
        cam_name = cam_dir.name
        scene_name = scene_dir.name
        split_name = split_dir.name
        if not cam_name.startswith("c") or not scene_name.startswith("S"):
            continue
        if split_name not in ALLOWED_SPLITS:
            skipped_splits.add(split_name)
            continue
        flat_name = f"{scene_name}_{cam_name}"
        target = CITYFLOW_DIR / flat_name
        if not target.exists():
            shutil.move(str(cam_dir), str(target))
            moved += 1

    if moved == 0:
        for split_name in sorted(ALLOWED_SPLITS):
            split_dir = Path(f"/tmp/{split_name}")
            if not split_dir.exists():
                continue
            for vdo_path in sorted(split_dir.rglob("vdo.avi")):
                cam_dir = vdo_path.parent
                scene_dir = cam_dir.parent
                if not cam_dir.name.startswith("c") or not scene_dir.name.startswith("S"):
                    continue
                flat_name = f"{scene_dir.name}_{cam_dir.name}"
                target = CITYFLOW_DIR / flat_name
                if not target.exists():
                    shutil.move(str(cam_dir), str(target))
                    moved += 1

    if staging.exists():
        shutil.rmtree(staging, ignore_errors=True)
    for extra_dir in [Path("/tmp/train"), Path("/tmp/test"), Path("/tmp/validation")]:
        if extra_dir.exists():
            shutil.rmtree(extra_dir, ignore_errors=True)
    for extra_file in [Path("/tmp/ReadMe.txt"), Path("/tmp/list_cam.txt")]:
        if extra_file.exists():
            extra_file.unlink()
    for license_file in Path("/tmp").glob("Dataset License*"):
        license_file.unlink(missing_ok=True)
    for extra_dir in [Path("/tmp/cam_framenum"), Path("/tmp/cam_loc"), Path("/tmp/cam_timestamp"), Path("/tmp/eval")]:
        if extra_dir.exists():
            shutil.rmtree(extra_dir, ignore_errors=True)

    print(json.dumps({
        "cityflow_dir": str(CITYFLOW_DIR),
        "flattened_cameras": len([d for d in CITYFLOW_DIR.iterdir() if d.is_dir()]),
        "skipped_splits": sorted(skipped_splits),
    }, indent=2))
    return CITYFLOW_DIR


def collect_camera_assets(data_root: Path):
    cameras = []
    cam_gt_paths = {}
    cam_vid_paths = {}
    for cam_dir in sorted(data_root.iterdir()):
        if not cam_dir.is_dir() or "_c" not in cam_dir.name:
            continue
        gt = find_gt(cam_dir)
        vid = find_video(cam_dir)
        if gt is None or vid is None:
            continue
        cameras.append(cam_dir.name)
        cam_gt_paths[cam_dir.name] = gt
        cam_vid_paths[cam_dir.name] = vid
    if cameras:
        return cameras, cam_gt_paths, cam_vid_paths

    for vdo_path in sorted(data_root.rglob("vdo.avi")):
        cam_dir = vdo_path.parent
        if cam_dir.name.startswith("c") and cam_dir.parent.name.startswith("S"):
            split_name = cam_dir.parent.parent.name
            if split_name not in ALLOWED_SPLITS:
                continue
            flat_name = f"{cam_dir.parent.name}_{cam_dir.name}"
        else:
            continue
        gt = find_gt(cam_dir)
        if gt is None:
            continue
        cameras.append(flat_name)
        cam_gt_paths[flat_name] = gt
        cam_vid_paths[flat_name] = vdo_path

    cameras = sorted(set(cameras))
    if not cameras:
        raise RuntimeError(f"No cameras with GT+video found under {data_root}")
    return cameras, cam_gt_paths, cam_vid_paths


def load_gt(gt_path: Path):
    rows = []
    with gt_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, tid = int(parts[0]), int(parts[1])
            x, y, w, h = map(int, parts[2:6])
            rows.append((frame, tid, x, y, w, h))
    return rows


def extract_crops_from_camera(cam_name: str, gt_file: Path, vid_file: Path, crop_dir: Path, max_crops: int, min_area: int):
    gt_rows = load_gt(gt_file)
    if not gt_rows:
        print(f"  {cam_name}: empty GT file")
        return {}

    id_dets = defaultdict(list)
    for frame, tid, x, y, w, h in gt_rows:
        id_dets[tid].append((frame, x, y, w, h))

    frame_to_dets = defaultdict(list)
    for tid, dets in id_dets.items():
        dets = sorted(dets, key=lambda item: item[0])
        if len(dets) <= max_crops:
            sampled = dets
        else:
            step = len(dets) / max_crops
            sampled = [dets[int(i * step)] for i in range(max_crops)]
        for frame, x, y, w, h in sampled:
            if w * h >= min_area and w >= MIN_BBOX_SIDE and h >= MIN_BBOX_SIDE:
                frame_to_dets[frame].append((tid, x, y, w, h))

    if not frame_to_dets:
        return {}

    crop_dir.mkdir(parents=True, exist_ok=True)
    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(vid_file))
    current_frame = 0
    target_frames = sorted(frame_to_dets.keys())
    target_idx = 0

    while target_idx < len(target_frames):
        ok, img = cap.read()
        if not ok:
            break
        current_frame += 1
        if current_frame < target_frames[target_idx]:
            continue
        if current_frame > target_frames[target_idx]:
            while target_idx < len(target_frames) and target_frames[target_idx] < current_frame:
                target_idx += 1
            if target_idx >= len(target_frames) or current_frame != target_frames[target_idx]:
                continue

        height, width = img.shape[:2]
        for tid, x, y, w, h in frame_to_dets[current_frame]:
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + w), min(height, y + h)
            if x2 - x1 < MIN_BBOX_SIDE or y2 - y1 < MIN_BBOX_SIDE:
                continue
            crop = img[y1:y2, x1:x2]
            fname = f"{tid:04d}_{cam_name}_f{current_frame:06d}.jpg"
            fpath = crop_dir / fname
            cv2.imwrite(str(fpath), crop)
            crops[tid].append(str(fpath))
        target_idx += 1

    cap.release()
    print(f"  {cam_name}: {sum(len(paths) for paths in crops.values())} crops from {len(crops)} vehicles")
    return dict(crops)


def load_existing_raw_crops(raw_dir: Path):
    all_crops = defaultdict(lambda: defaultdict(list))
    for fpath in sorted(raw_dir.glob("*.jpg")):
        parts = fpath.stem.split("_")
        if len(parts) < 4:
            continue
        tid = int(parts[0])
        cam_name = parts[1] + "_" + parts[2]
        all_crops[tid][cam_name].append(str(fpath))
    return {tid: dict(cam_map) for tid, cam_map in all_crops.items()}


def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def frame_number_from_name(path: str) -> int:
    stem = Path(path).stem
    marker = stem.rsplit("_f", 1)
    return int(marker[1]) if len(marker) == 2 and marker[1].isdigit() else 0


def build_prepared_splits(all_crops, root: Path):
    if root.exists():
        shutil.rmtree(root, ignore_errors=True)
    for split in ["train", "query", "gallery"]:
        (root / split).mkdir(parents=True, exist_ok=True)

    split_stats = {split: 0 for split in ["train", "query", "gallery"]}
    for tid, cam_map in sorted(all_crops.items()):
        pid_dir_name = f"ID_{tid:04d}"
        sorted_cam_items = sorted((cam_name, sorted(paths, key=frame_number_from_name)) for cam_name, paths in cam_map.items())
        eval_cam_names = [cam_name for cam_name, _ in sorted_cam_items[: max(1, min(len(sorted_cam_items), QUERY_CAMS_PER_ID))]]

        for cam_name, paths in sorted_cam_items:
            for path_str in paths:
                src = Path(path_str)
                dst = root / "train" / pid_dir_name / src.name
                link_or_copy(src, dst)
                split_stats["train"] += 1

            query_count = 0
            if cam_name in eval_cam_names and len(paths) > 1:
                query_count = min(QUERY_FRAMES_PER_CAM, len(paths) - 1)
            query_paths = paths[:query_count]
            gallery_paths = paths[query_count:]

            for path_str in query_paths:
                src = Path(path_str)
                dst = root / "query" / pid_dir_name / src.name
                link_or_copy(src, dst)
                split_stats["query"] += 1
            for path_str in gallery_paths:
                src = Path(path_str)
                dst = root / "gallery" / pid_dir_name / src.name
                link_or_copy(src, dst)
                split_stats["gallery"] += 1

    return split_stats


def parse_prepared_cityflow(root: Path):
    train, query, gallery = [], [], []
    for split_name, split_list in [("train", train), ("query", query), ("gallery", gallery)]:
        split_dir = root / split_name
        if not split_dir.is_dir():
            raise FileNotFoundError(f"Prepared CityFlowV2 split not found: {split_dir}")
        for pid_dir in sorted(split_dir.iterdir()):
            if not pid_dir.is_dir() or not pid_dir.name.startswith("ID_"):
                continue
            for img_path in sorted(pid_dir.rglob("*.jpg")):
                parts = img_path.stem.split("_")
                if len(parts) < 4:
                    continue
                pid = int(parts[0])
                cam_name = parts[1] + "_" + parts[2]
                split_list.append((str(img_path), pid, cam_name))

    all_cams = sorted({cam for _, _, cam in train + query + gallery})
    cam2id = {cam: idx for idx, cam in enumerate(all_cams)}
    train = [(path, pid, cam2id[cam]) for path, pid, cam in train]
    query = [(path, pid, cam2id[cam]) for path, pid, cam in query]
    gallery = [(path, pid, cam2id[cam]) for path, pid, cam in gallery]

    train_pids = sorted({pid for _, pid, _ in train})
    pid2label = {pid: label for label, pid in enumerate(train_pids)}
    train = [(path, pid2label[pid], cam) for path, pid, cam in train]
    return train, query, gallery


if prepared_reid_exists(REID_ROOT):
    prep_stats = split_counts(REID_ROOT)
    print(f"Reusing prepared ReID splits from {REID_ROOT}")
else:
    data_root = ensure_cityflow_download()
    cameras, cam_gt_paths, cam_vid_paths = collect_camera_assets(data_root)
    print(f"Using {len(cameras)} cameras with GT+video")

    existing_raw = list(RAW_CROP_DIR.glob("*.jpg")) if RAW_CROP_DIR.exists() else []
    if len(existing_raw) > 100:
        print(f"Reusing {len(existing_raw)} raw crops from {RAW_CROP_DIR}")
        all_crops = load_existing_raw_crops(RAW_CROP_DIR)
    else:
        RAW_CROP_DIR.mkdir(parents=True, exist_ok=True)
        all_crops = defaultdict(lambda: defaultdict(list))
        print("Extracting crops from videos...")
        for cam_name in cameras:
            cam_crops = extract_crops_from_camera(
                cam_name,
                cam_gt_paths[cam_name],
                cam_vid_paths[cam_name],
                RAW_CROP_DIR,
                MAX_CROPS_PER_ID_CAM,
                MIN_AREA,
            )
            for tid, paths in cam_crops.items():
                all_crops[tid][cam_name].extend(paths)
        all_crops = {tid: dict(cam_map) for tid, cam_map in all_crops.items()}

    prep_stats = build_prepared_splits(all_crops, REID_ROOT)
    if RAW_CROP_DIR.exists():
        shutil.rmtree(RAW_CROP_DIR, ignore_errors=True)
    if CITYFLOW_DIR.exists() and str(CITYFLOW_DIR).startswith("/tmp/"):
        shutil.rmtree(CITYFLOW_DIR, ignore_errors=True)

if not SBS_WEIGHTS_PATH.exists():
    print(f"Downloading FastReID SBS weights -> {SBS_WEIGHTS_PATH}")
    urllib.request.urlretrieve(SBS_WEIGHTS_URL, SBS_WEIGHTS_PATH)

print(json.dumps({
    "reid_root": str(REID_ROOT),
    "split_counts": prep_stats,
    "weights_path": str(SBS_WEIGHTS_PATH),
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "device": DEVICE,
}, indent=2))

In [ ]:
from torch.utils.data import DataLoader

from src.training.datasets import PKSampler, ReIDDataset, build_test_transforms, build_train_transforms

train_data, query_data, gallery_data = parse_prepared_cityflow(REID_ROOT)
NUM_CLASSES = len({pid for _, pid, _ in train_data})
NUM_CAMERAS = len({cam for _, _, cam in (train_data + query_data + gallery_data)})
if NUM_EXPECTED_CLASSES is not None:
    assert NUM_CLASSES == NUM_EXPECTED_CLASSES, f"Expected {NUM_EXPECTED_CLASSES} training IDs, found {NUM_CLASSES}"
assert NUM_CLASSES > 0, "Train split is empty"
assert query_data and gallery_data, "Query/gallery split is empty"

train_tf = build_train_transforms(
    height=IMG_SIZE[0],
    width=IMG_SIZE[1],
    random_erasing_prob=0.5,
    color_jitter=False,
)
test_tf = build_test_transforms(height=IMG_SIZE[0], width=IMG_SIZE[1])

train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=PKSampler(train_data, p=P_IDS, k=K_INSTANCES),
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
query_loader = DataLoader(
    query_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    persistent_workers=NUM_WORKERS > 0,
)
gallery_loader = DataLoader(
    gallery_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    persistent_workers=NUM_WORKERS > 0,
)

print(json.dumps({
    "train_images": len(train_data),
    "query_images": len(query_data),
    "gallery_images": len(gallery_data),
    "num_classes": NUM_CLASSES,
    "num_cameras": NUM_CAMERAS,
    "train_batches": len(train_loader),
    "query_batches": len(query_loader),
    "gallery_batches": len(gallery_loader),
}, indent=2))

In [ ]:
import torch.nn as nn

from src.training.model import ReIDModelResNet50IBN

def unwrap_checkpoint_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        if 'state_dict' in checkpoint:
            return checkpoint['state_dict']
        if 'model' in checkpoint:
            return checkpoint['model']
        return checkpoint
    raise TypeError(f'Unsupported checkpoint type: {type(checkpoint)!r}')

def remap_fastreid_sbs_r50_ibn_state_dict(state_dict):
    remapped = {}
    skip_prefixes = ('pixel_mean', 'pixel_std', 'heads.classifier')
    for key, value in state_dict.items():
        key = key.replace('module.', '', 1)
        if key.startswith(skip_prefixes):
            continue
        if key.startswith('backbone.NL_'):
            continue
        if key == 'heads.pool_layer.p':
            remapped['pool.p'] = value
            continue
        if key.startswith('heads.bottleneck.'):
            suffix = key[len('heads.bottleneck.'): ]
            if suffix.startswith('0.'):
                suffix = suffix[2:]
            remapped[f'bottleneck.{suffix}'] = value
            continue
        if key == 'heads.bnneck.num_batches_tracked':
            remapped['bottleneck.num_batches_tracked'] = value
            continue
        if key.startswith('backbone.'):
            remapped[key] = value
    return remapped

model = ReIDModelResNet50IBN(
    num_classes=NUM_CLASSES,
    last_stride=1,
    pretrained=False,
    gem_p=GEM_P,
    eval_feature='global',
)
checkpoint = torch.load(SBS_WEIGHTS_PATH, map_location='cpu', weights_only=False)
state_dict = remap_fastreid_sbs_r50_ibn_state_dict(unwrap_checkpoint_state_dict(checkpoint))
missing, unexpected = model.load_state_dict(state_dict, strict=False)
nn.init.normal_(model.classifier.weight, std=0.001)
model.pool.p.requires_grad_(False)
model = model.to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
base_model = model.module if hasattr(model, 'module') else model
critical_missing = [key for key in missing if not key.startswith('classifier')]

print(json.dumps({
    'loaded_tensors': len(state_dict),
    'critical_missing': critical_missing[:20],
    'unexpected': unexpected[:20],
    'pool_p': float(base_model.pool.p.detach().cpu().item()),
    'pool_p_frozen': not base_model.pool.p.requires_grad,
    'feat_dim': base_model.feat_dim,
    'eval_feature': base_model.eval_feature,
}, indent=2))

In [ ]:
from src.training.losses import CenterLoss, CrossEntropyLabelSmooth, TripletLoss
from src.training.train_reid import build_cosine_scheduler

id_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=LABEL_SMOOTHING).to(DEVICE)
triplet_loss_fn = TripletLoss(margin=TRIPLET_MARGIN, soft_margin=True).to(DEVICE)
center_loss_fn = CenterLoss(NUM_CLASSES, 2048).to(DEVICE)

def unwrap_model(current_model):
    return current_model.module if hasattr(current_model, 'module') else current_model

def set_backbone_trainable(current_model, trainable: bool):
    raw_model = unwrap_model(current_model)
    for param in raw_model.backbone.parameters():
        param.requires_grad_(trainable)
    raw_model.pool.p.requires_grad_(False)

def build_optimizers(current_model):
    raw_model = unwrap_model(current_model)
    optimizer = torch.optim.AdamW(
        [
            {'params': raw_model.backbone.parameters(), 'lr': BACKBONE_LR},
            {'params': raw_model.bottleneck.parameters(), 'lr': HEAD_LR},
            {'params': raw_model.classifier.parameters(), 'lr': HEAD_LR},
        ],
        betas=(0.9, 0.999),
        weight_decay=WEIGHT_DECAY,
    )
    center_optimizer = torch.optim.SGD(center_loss_fn.parameters(), lr=0.5)
    scheduler = build_cosine_scheduler(
        optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=EPOCHS,
        eta_min=1e-7,
        start_factor=0.1,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    return optimizer, center_optimizer, scheduler, scaler

set_backbone_trainable(model, False)
optimizer, center_optimizer, scheduler, scaler = build_optimizers(model)
print(json.dumps({
    'triplet_soft_margin': True,
    'triplet_weight': TRIPLET_WEIGHT,
    'center_weight': CENTER_WEIGHT,
    'center_start_epoch': CENTER_START_EPOCH,
    'freeze_backbone_epochs': FREEZE_BACKBONE_EPOCHS,
    'optimizer_lrs': [group['lr'] for group in optimizer.param_groups],
}, indent=2))

In [ ]:
from src.training.evaluate_reid import evaluate_reid

def evaluate_model(current_model):
    mAP, cmc, _, _ = evaluate_reid(
        current_model,
        query_loader,
        gallery_loader,
        device=DEVICE,
        rerank=False,
    )
    return {
        'mAP': float(mAP),
        'rank1': float(cmc[0]) if len(cmc) > 0 else 0.0,
        'rank5': float(cmc[4]) if len(cmc) > 4 else 0.0,
        'rank10': float(cmc[9]) if len(cmc) > 9 else 0.0,
    }

baseline_metrics = evaluate_model(model)
print(json.dumps({'baseline_metrics': baseline_metrics}, indent=2))

In [ ]:
def save_training_state(epoch, best_mAP, best_epoch, history_records):
    torch.save({
        'epoch': epoch,
        'model': unwrap_model(model).state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_mAP': best_mAP,
        'best_epoch': best_epoch,
        'history': history_records,
    }, LAST_CHECKPOINT_PATH)

def train_one_epoch(epoch):
    raw_model = unwrap_model(model)
    if epoch == FREEZE_BACKBONE_EPOCHS + 1:
        set_backbone_trainable(model, True)
        print(f'Unfroze backbone at epoch {epoch}')

    model.train()
    use_center = epoch >= CENTER_START_EPOCH
    running = {'loss': 0.0, 'id_loss': 0.0, 'tri_loss': 0.0, 'cen_loss': 0.0}

    for batch_idx, (imgs, pids, _, _) in enumerate(train_loader, start=1):
        imgs = imgs.to(DEVICE, non_blocking=USE_AMP)
        pids = pids.to(DEVICE).long()

        optimizer.zero_grad(set_to_none=True)
        if use_center:
            center_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=USE_AMP):
            cls_score, global_feat, _bn_feat = model(imgs)
            loss_id = id_loss_fn(cls_score, pids)
            loss_tri = triplet_loss_fn(global_feat, pids)
            loss_cen_raw = center_loss_fn(global_feat.float(), pids) if use_center else torch.tensor(0.0, device=global_feat.device)
            loss = loss_id + (TRIPLET_WEIGHT * loss_tri) + (CENTER_WEIGHT * loss_cen_raw)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if use_center:
                scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            if use_center:
                for param in center_loss_fn.parameters():
                    if param.grad is not None:
                        param.grad.data *= 1.0 / max(CENTER_WEIGHT, 1e-12)
            scaler.step(optimizer)
            if use_center:
                scaler.step(center_optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            if use_center:
                for param in center_loss_fn.parameters():
                    if param.grad is not None:
                        param.grad.data *= 1.0 / max(CENTER_WEIGHT, 1e-12)
            optimizer.step()
            if use_center:
                center_optimizer.step()

        running['loss'] += float(loss.detach().item())
        running['id_loss'] += float(loss_id.detach().item())
        running['tri_loss'] += float(loss_tri.detach().item())
        running['cen_loss'] += float(loss_cen_raw.detach().item()) if use_center else 0.0

        if batch_idx % 10 == 0 or batch_idx == len(train_loader):
            avg_loss = running['loss'] / batch_idx
            avg_id = running['id_loss'] / batch_idx
            avg_tri = running['tri_loss'] / batch_idx
            avg_cen = running['cen_loss'] / batch_idx
            print(
                f'Epoch {epoch:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | '
                f'loss={avg_loss:.4f} id={avg_id:.4f} tri={avg_tri:.4f} cen={avg_cen:.4f} '
                f"lr_backbone={optimizer.param_groups[0]['lr']:.2e} lr_head={optimizer.param_groups[1]['lr']:.2e}"
            )

    scheduler.step()
    num_batches = max(len(train_loader), 1)
    return {key: value / num_batches for key, value in running.items()}

In [ ]:
history = []
best_mAP = -1.0
best_epoch = 0
best_metrics = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(epoch)
    record = {
        'epoch': epoch,
        'lr_backbone': float(optimizer.param_groups[0]['lr']),
        'lr_head': float(optimizer.param_groups[1]['lr']),
        **{key: float(value) for key, value in train_metrics.items()},
    }

    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        eval_metrics = evaluate_model(model)
        record.update(eval_metrics)
        print(
            f"Eval epoch {epoch:03d} | mAP={eval_metrics['mAP']:.4f} "
            f"R1={eval_metrics['rank1']:.4f} R5={eval_metrics['rank5']:.4f} R10={eval_metrics['rank10']:.4f}"
        )
        if eval_metrics['mAP'] > best_mAP:
            best_mAP = eval_metrics['mAP']
            best_epoch = epoch
            best_metrics = dict(eval_metrics)
            torch.save(unwrap_model(model).state_dict(), BEST_MODEL_PATH)
            print(f'Saved new best checkpoint -> {BEST_MODEL_PATH}')

    history.append(record)
    with HISTORY_PATH.open('w', encoding='utf-8') as handle:
        json.dump(history, handle, indent=2)
    save_training_state(epoch, best_mAP, best_epoch, history)

assert BEST_MODEL_PATH.exists(), f'Missing best checkpoint: {BEST_MODEL_PATH}'
print(json.dumps({
    'best_epoch': best_epoch,
    'best_mAP': best_mAP,
    'history_path': str(HISTORY_PATH),
    'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
}, indent=2))

In [ ]:
best_state = torch.load(BEST_MODEL_PATH, map_location='cpu', weights_only=False)
unwrap_model(model).load_state_dict(best_state, strict=True)
final_eval_metrics = evaluate_model(model)

final_payload = {
    'reid_root': str(REID_ROOT),
    'weights_path': str(SBS_WEIGHTS_PATH),
    'best_model_path': str(BEST_MODEL_PATH),
    'best_epoch': best_epoch,
    'best_metrics': best_metrics if best_metrics is not None else final_eval_metrics,
    'final_eval_metrics': final_eval_metrics,
    'baseline_metrics': baseline_metrics,
    'config': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'p_ids': P_IDS,
        'k_instances': K_INSTANCES,
        'epochs': EPOCHS,
        'warmup_epochs': WARMUP_EPOCHS,
        'freeze_backbone_epochs': FREEZE_BACKBONE_EPOCHS,
        'backbone_lr': BACKBONE_LR,
        'head_lr': HEAD_LR,
        'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING,
        'triplet_soft_margin': True,
        'triplet_weight': TRIPLET_WEIGHT,
        'center_weight': CENTER_WEIGHT,
        'center_start_epoch': CENTER_START_EPOCH,
        'eval_every': EVAL_EVERY,
    },
}
with METRICS_PATH.open('w', encoding='utf-8') as handle:
    json.dump(final_payload, handle, indent=2)

state_dict = torch.load(BEST_MODEL_PATH, map_location='cpu', weights_only=False)
required_prefixes = [
    'backbone.conv1',
    'backbone.layer1',
    'backbone.layer2',
    'backbone.layer3',
    'backbone.layer4',
    'pool.p',
    'bottleneck',
    'classifier',
]
prefix_hits = {prefix: any(key.startswith(prefix) for key in state_dict) for prefix in required_prefixes}
assert all(prefix_hits.values()), prefix_hits

unwrap_model(model).eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE[0], IMG_SIZE[1], device=DEVICE)
    embeddings = unwrap_model(model)(dummy)
assert tuple(embeddings.shape) == (2, 2048), tuple(embeddings.shape)

print(json.dumps({
    'best_model_path': str(BEST_MODEL_PATH),
    'parameter_tensors': len(state_dict),
    'prefix_hits': prefix_hits,
    'eval_embedding_shape': list(embeddings.shape),
    'history_path': str(HISTORY_PATH),
    'metrics_path': str(METRICS_PATH),
}, indent=2))